# 06 - Generation And API (Hue Foods RAG MVP)

Notebook này chạy **full runtime path thật** của Phase 6 qua API: câu hỏi -> retrieval thật (Qdrant + E5) -> ContextBuilder thật -> OpenAI generator thật (`gpt-5.4-nano` qua OpenAI Agents SDK) -> JSON sources.

**Milestone 6.1 - Baseline Lifecycle Hardening**

- FastAPI lifespan build toàn bộ retrieval stack một lần: Qdrant read-only preflight -> warm E5 bằng một query nội bộ cố định -> (BM25 fit + MiniLM load/prediction chỉ khi profile cần) -> publish stack và `retrieval_ready=true`.
- Model load xảy ra ở **startup**, không còn ở request/health đầu tiên.
- Bằng chứng warm-up: cache E5/MiniLM được chụp **trước mọi request** (Evidence A), sau `/health` (A2) và sau first `/api/chat` (B). Assert cuối đảm bảo misses không tăng sau first retrieval, và `runtime snapshot` cho thấy profile/model thật.

**Prerequisite**

- Khởi động Jupyter từ terminal đã export `OPENAI_API_KEY` vào environment (notebook không đọc `.env`, không dùng `load_dotenv`). Nếu thiếu key, notebook fail actionable ngay - không có fallback.
- Qdrant local đang chạy (collection `hue_foods_e5_small_384`, 572 points) và E5 đã cache.

**Chi phí**

- Mỗi Run All gọi **đúng 1** OpenAI call (`POST /api/chat`). Chi phí ước tính dưới 0,001 USD mỗi call (dựa trên smoke 2026-08-13: ~1.100-1.400 input tokens, ~100-500 output tokens).

**Kết quả mong đợi khi Run All**

- `/health` trả `ok` với các component ready.
- `/api/chat` trả HTTP 200: answer tiếng Việt grounded, `sources` theo context order, `retrieval_debug` với profile/model thật.
- `startup_seconds` là số thực (lần chạy đầu có thể vài chục giây khi E5 chưa cache; lần sau nhỏ hơn).
- Evidence A: `E5 cache.misses >= 1` (E5 đã load trong startup); `MiniLM cache.misses >= 1` chỉ khi `active_profile=hybrid_rerank`, nếu không thì `= 0` (profile-scoped).
- Evidence B so A: misses của E5 và MiniLM **không tăng** sau first `/api/chat`; dòng `PASS: cache misses unchanged by first retrieval` xuất hiện.
- Nếu có lỗi provider, response trả safe error shape; notebook không retry.

Lần chạy đầu tiên có thể mất vài chục giây vì E5 load từ cache vào memory.

In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Khong tim thay thu muc backend/. Hay mo notebook nay tu repo root "
        "hoac tu thu muc notebooks/."
    )
print(f"backend on path: {sys.path[0]}")


## Kiểm tra key và offline mode

Cell dưới chỉ kiểm tra **presence** của `OPENAI_API_KEY` (không in giá trị) và đặt `HF_HUB_OFFLINE=1` để E5/MiniLM chỉ dùng local cache. Nếu key thiếu, notebook dừng với thông báo rõ ràng.


In [ ]:
import os

os.environ["HF_HUB_OFFLINE"] = "1"

has_key = bool(os.environ.get("OPENAI_API_KEY", "").strip())
print("OPENAI_API_KEY present:", has_key)
if not has_key:
    raise RuntimeError(
        "Thieu OPENAI_API_KEY trong environment. Hay khoi dong Jupyter tu "
        "terminal da export key (export OPENAI_API_KEY=...) roi chay lai."
    )


## Câu hỏi

Người dùng chỉ cần sửa đúng một biến dưới đây rồi Run All. Không cần tự tạo chunk_id, evidence JSON hay available_source_ids - API tự chạy retrieval và build context.


In [ ]:
question = "Bún bò Huế có gì đặc biệt?"
print("question:", question)


## Gọi `/api/chat` qua app thật

Cell dưới dùng FastAPI `TestClient` với app thật và lifespan thật: lifespan build retrieval stack một lần (Qdrant + E5 thật, và MiniLM/BM25 theo profile), rồi `/health` đọc cached readiness và `POST /api/chat` chạy đúng 1 retrieval + 1 OpenAI call. In ra answer, sources projection, session_id và retrieval_debug - không in prompt, raw SDK response, header hay token.

**Cache evidence Warm-up (Milestone 6.1):**

- Evidence **A** chụp ngay sau lifespan startup **trước mọi request**;
- Evidence **A2** chụp sau `/health` (chỉ đọc cached readiness);
- Evidence **B** chụp sau first `/api/chat`.
- Assert ở cuối cell: `E5 misses` và `MiniLM misses` của B phải **bằng** A — nghĩa là first retrieval không tạo model cache miss mới (component đã warm ở startup). Assert fail nghĩa là warm-up không chạy ở startup, notebook dừng rõ ràng.
- `MiniLM misses` chỉ có thể khác 0 khi `active_profile=hybrid_rerank`; hai profile còn lại để 0.

In [ ]:
import time

from fastapi.testclient import TestClient

from api.app import app

# Cache evidence chụp "trước" mọi request: nếu model load ở request đầu tiên
# (first retrieval) thay vì startup, misses dưới đây sẽ tăng và assert bên
# dưới sẽ fail ngay - không có fake fallback.
from embedding.embedder import _get_model
from reranking.models import cross_encoder as rerank_module

# Lifespan của app chạy bên trong context của TestClient: thời gian từ lúc vào
# context đến khi request đầu tiên thực thi là thời gian startup thật
# (build_retrieval_stack + warm-up theo active profile trong settings).
t0 = time.monotonic()
with TestClient(app) as client:
    startup_seconds = round(time.monotonic() - t0, 1)

    # Evidence A: ngay sau lifespan startup, trước /health và /api/chat.
    cache_after_startup = {
        "e5": _get_model.cache_info(),
        "minilm": rerank_module._get_cross_encoder.cache_info(),
    }
    print("cache evidence AFTER startup (before any request):")
    print("  E5 cache:", cache_after_startup["e5"])
    print("  MiniLM cache:", cache_after_startup["minilm"])

    health = client.get("/health")
    print("health status:", health.status_code)
    print("health body:", health.json())

    # Evidence A2: sau /health - /health chỉ đọc cached readiness trong
    # app.state, không tải model hay gọi service nào.
    cache_after_health = {
        "e5": _get_model.cache_info(),
        "minilm": rerank_module._get_cross_encoder.cache_info(),
    }
    print("cache AFTER /health (cached readiness):")
    print("  E5 misses:", cache_after_health["e5"].misses,
          "| MiniLM misses:", cache_after_health["minilm"].misses)

    started = time.monotonic()
    response = client.post("/api/chat", json={"query": question})
    elapsed = round(time.monotonic() - started, 1)
    print("chat status:", response.status_code)
    print("elapsed seconds:", elapsed)

    body = response.json()
    if response.status_code == 200:
        print("answer:", body["answer"])
        print("sources:")
        for source in body["sources"]:
            print("  -", source["title"], "|", source["section"],
                  "| score:", source["score"], "|", source["chunk_id"])
        print("session_id:", body["session_id"])
        print("retrieval_debug:", body["retrieval_debug"])
    else:
        print("error body (safe shape):", body)

    # Evidence B: sau first /api/chat (retrieval + generation đã chạy).
    cache_after_chat = {
        "e5": _get_model.cache_info(),
        "minilm": rerank_module._get_cross_encoder.cache_info(),
    }

print("cache evidence AFTER first /api/chat:")
print("  E5 cache:", cache_after_chat["e5"])
print("  MiniLM cache:", cache_after_chat["minilm"])
assert cache_after_chat["e5"].misses == cache_after_startup["e5"].misses, (
    "first retrieval added a new E5 model cache miss; warm-up must happen at "
    "startup"
)
assert cache_after_chat["minilm"].misses == cache_after_startup["minilm"].misses, (
    "first retrieval added a new MiniLM model cache miss; warm-up must happen "
    "at startup"
)
print("PASS: cache misses unchanged by first retrieval - components were "
      "fully warm at startup")

print("warm-up summary:")
print("  startup_seconds:", startup_seconds)
print("  runtime snapshot:", app.state.runtime)

## Checklist xác nhận Phase 6 (Milestone 6.1)

1. `OPENAI_API_KEY present: True` được in ra (không bao giờ in giá trị key).
2. `/health` trả `ok` với `qdrant: ready`, `retrieval: ready`, `generator: configured`.
3. `/api/chat` trả HTTP 200 với answer tiếng Việt, `sources` hợp lệ theo context order và `retrieval_debug` đúng profile/model thật.
4. Nếu provider trả structured output không hợp lệ, response là HTTP 502 với safe error shape - notebook không retry.
5. Đúng 1 OpenAI call cho mỗi Run All.
6. Evidence A: ngay sau startup (trước mọi request), `E5 cache.misses >= 1` — E5 đã load lúc startup; `MiniLM cache.misses >= 1` chỉ khi `active_profile=hybrid_rerank`, nếu không thì `= 0` (profile-scoped).
7. Evidence B so A: `E5 misses` và `MiniLM misses` **không tăng** sau first `/api/chat` — pass in ra "PASS: cache misses unchanged by first retrieval" và assert ở cuối cell không fail.
8. `startup_seconds` là số thực và `runtime snapshot` cho thấy active profile + embedding model từ startup snapshot (immutable).
9. Hai lần gọi `/health` sau startup có latency nhỏ và không tạo dependency work mới (chỉ đọc cached readiness).